In [1]:
import json
import pandas as pd
import numpy as np
import psycopg2
import pyodbc
import mariadb

In [2]:
df = pd.read_excel("inputFiles/batch-approval-batch_9507 (14).xlsx",sheet_name="Scoring Results",header=1)

In [3]:
comparison_df = pd.read_csv("inputFiles/KALRO Test protocol.csv")    

In [4]:
conn_lims = pyodbc.connect("Driver={SQL Server};"
                            "Server=192.168.5.18\CROPNUT;"
                            "Database=cropnuts;"
                            "uid=thomasTsuma;pwd=GR^KX$uRe9#JwLc6")
cursor_lims = conn_lims.cursor()

In [5]:
df

,Sample Code,Batch No,Sample Date,Report Date,Farmer Name,Farmer Phone,Sampler Name,Crop,Barcode,Location,...,Exchangeable Acidity,EC (Salts),Sodium,aluminium,Sulphur,Potassium,Soil Texture,Zinc Calculated,Exchangeable K Calculated,Region
0,TEST-DS1-1660,109507,13-03-23,27-09-24,Farmer 5,715341345,ttu,Maize,TEST-DS1-1660-1,Kiambu/Githunguri,...,0.0500,161.9686,68.4058,0,13.8609,459.8820,LSand,10.0806,459.8820,Central
1,TEST-DS1-1661,109507,03-03-23,27-09-24,NORTSU EGA,246190470,Devine Foli,Maize,TEST-DS1-1661-1,Kiambu/Githunguri,...,0.0500,133.3273,28.8970,0,20.8689,99.6635,Sand,3.4790,99.6635,Central
2,TEST-DS1-1663,109507,20-03-23,27-09-24,Mwakisha Mugendi (Mwakisha Mugendi - Mwakisha ...,713296124,Sally Munyao,Maize,TEST-DS1-1663-1,Kiambu/Githunguri,...,0.0975,233.1011,239.6181,0,54.4275,1997.5518,Clay,11.3119,1997.5518,Central
3,TEST-DS1-1664,109507,11-10-23,27-09-24,Moses Gitau. Kamau (Moses Gitau Kamau - Moses ...,70041001881,Priscillah Wangeci,Maize,TEST-DS1-1664-1,Kiambu/Githunguri,...,0.0500,123.5313,36.7639,0,9.4655,122.2531,LSand,3.4371,122.2531,Central
4,TEST-DS1-1665,109507,20-03-23,27-09-24,Tettey Dogbey,551543019,Samuel Anthonio,Maize,TEST-DS1-1665-1,Kiambu/Githunguri,...,0.2795,141.1284,43.2046,0,12.4112,58.9581,SLoam,5.0478,58.9581,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,TEST-DS1-1767,109507,29-09-23,27-09-24,Samweli Tibenda Samweli,624019165,Leona Cyrilo,Tomatoes (Open field),TEST-DS1-1767-66,Kiambu/Githunguri,...,0.3441,152.2636,77.1235,0,15.7202,132.9937,LSand,1.8700,132.9937,Central
71,TEST-DS1-1768,109507,07-03-23,27-09-24,Betty Obonyo(BLOCK B),722255206,Philip Kilenyi,Tomatoes (Open field),TEST-DS1-1768-66,Kiambu/Githunguri,...,0.1605,149.7990,179.7527,0,29.5243,959.0062,Clay,4.5223,959.0062,Central
72,TEST-DS1-1769,109507,11-10-23,27-09-24,Moses Gitau. Kamau (Moses Gitau Kamau - Moses ...,70041001881,Priscillah Wangeci,Tomatoes (Open field),TEST-DS1-1769-66,Kiambu/Githunguri,...,0.0500,142.5406,37.5295,0,11.7796,319.0961,Loam,3.8514,319.0961,Central
73,TEST-DS1-1770,109507,31-03-23,27-09-24,Evans Kpesese,543771567,Sokode Fred,Tomatoes (Open field),TEST-DS1-1770-66,Kiambu/Githunguri,...,0.3592,132.8214,65.7896,0,17.1693,130.9087,SLoam,2.8709,130.9087,Central


In [6]:
config = pd.read_sql("""SELECT [Chemical_Config_Id]
       ,[chemical_code]
      ,[Spectral_Lod]
      ,[spectral_Decimal_places]
      ,[spectral_Significant_figure]
  FROM [cropnuts].[dbo].[Chemicals_Config]
  where [Type_Code]=4""",con=conn_lims)
chemicals = pd.read_sql("SELECT chemical_code, chemical_name FROM Chemicals",con=conn_lims)
config = pd.merge(config, chemicals,on="chemical_code",how="inner")
config.chemical_name = [i.lower().replace(" ","_").replace("(","").replace(")","").replace(".","") for i in config.chemical_name]
config= config[['chemical_name','spectral_Decimal_places','spectral_Significant_figure']]
config = config.set_index('chemical_name')
config = config.to_dict()

C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_34216\2924685361.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  config = pd.read_sql("""SELECT [Chemical_Config_Id]
C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_34216\2924685361.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chemicals = pd.read_sql("SELECT chemical_code, chemical_name FROM Chemicals",con=conn_lims)


In [7]:
def round_number(number, precision):
    if precision == 0.1:
        return round(number, 1)
    elif precision == 0.01:
        return round(number, 2)
    elif precision == 0.001:
        return round(number, 3)
    elif precision == 0.0001:
        return round(number, 4)
    else:
        return number

In [8]:
for column in df.columns:
    try:
        df[column] = [ float(str(i).replace("<","").replace(">","")) for i in df[column] ]
        print(column)
    except Exception as e:
        print("Failed:  ",column)
        print(e)
        
    if column in config['spectral_Decimal_places'].keys() and column != "phosphorus":
        print(column)
        print(df[column].astype(str).replace("<","").replace(">","").astype(float))
        if (config['spectral_Significant_figure'][column]) >= 0:
            df[column] = [ round_number(i,config['spectral_Significant_figure'][column]) for i in df[column] ]
            # round(decimals=int(config['spectral_Decimal_places'][column]))
        elif (config['spectral_Decimal_places'][column]) >= 0:
            df[column] = df[column].round(decimals=int(config['spectral_Decimal_places'][column]))
        else:
            continue


Failed:   Sample Code
could not convert string to float: 'TEST-DS1-1660'
Batch No
Failed:   Sample Date
could not convert string to float: '13-03-23'
Failed:   Report Date
could not convert string to float: '27-09-24'
Failed:   Farmer Name
could not convert string to float: 'Farmer 5'
Farmer Phone
Failed:   Sampler Name
could not convert string to float: 'ttu'
Failed:   Crop
could not convert string to float: 'Maize'
Failed:   Barcode
could not convert string to float: 'TEST-DS1-1660-1'
Failed:   Location
could not convert string to float: 'Kiambu/Githunguri'
Latitude
Longitude
Phone Number
Failing Elements
pH
Failed:   Available P
could not convert string to float: '30-120'
Exchangeable K
Calcium
Magnesium
Ca%
Mg%
Ca:Mg
Iron
Failed:   IronClass
could not convert string to float: 'Adequate'
Manganese
Failed:   ManganeseClass
could not convert string to float: 'Adequate'
Boron
Failed:   BoronClass
could not convert string to float: 'Adequate'
Copper
Failed:   CopperClass
could not conve

In [9]:
def classifyResults(X, cec, chemical, lim1, lim2, lim3, crop_code, client_id, chemical_code):   
     
    if chemical == 'potassium':
        potassium_veryhigh_guide, potassium_high_guide, potassium_sample_guide, potassium_low_guide, potassium_verylow_guide = cursor_lims.execute(
        f"""
            DECLARE @Critical_K FLOAT;
            select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
        
            DECLARE @Critical_Ca FLOAT;
            select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
        
            DECLARE @Critical_Mg FLOAT;
            select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
        
            DECLARE @interpretation_Code INT;
            select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
        
            DECLARE @veryhigh_guide FLOAT;
            select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
            select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
        
            DECLARE @high_guide FLOAT;
            select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
            select @high_guide = {cec} * @high_guide/100 *390;
            
            DECLARE @sample_guide FLOAT;
            select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
            select @sample_guide= {cec} *@sample_guide/100 *390
        
            DECLARE @low_guide FLOAT;
            select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
            select @low_guide= {cec} * @low_guide/100 * 390
        
            DECLARE @verylow_guide FLOAT;
            select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
            select @verylow_guide= {cec} * @verylow_guide/100 * 390
            
            select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        if X <= potassium_verylow_guide:
            return "very low"
        elif X <= potassium_low_guide:
            return "low"
        elif X <= potassium_high_guide:
            return "optimum"
        else:
            return "high"
    
    elif chemical == 'magnesium':
        magnesium_veryhigh_guide, magnesium_high_guide, magnesium_sample_guide, magnesium_low_guide, magnesium_verylow_guide = cursor_lims.execute(
        f"""
        DECLARE @Critical_K FLOAT;
        select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Ca FLOAT;
        select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Mg FLOAT;
        select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @interpretation_Code INT;
        select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
    
        DECLARE @veryhigh_guide FLOAT;
        select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
        select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
    
        DECLARE @high_guide FLOAT;
        select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
        select @high_guide = {cec} * @high_guide/100 *390;
        
        DECLARE @sample_guide FLOAT;
        select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
        select @sample_guide= {cec} *@sample_guide/100 *390
    
        DECLARE @low_guide FLOAT;
        select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
        select @low_guide= {cec} * @low_guide/100 * 390
    
        DECLARE @verylow_guide FLOAT;
        select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
        select @verylow_guide= {cec} * @verylow_guide/100 * 390
        
        select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        if X <= magnesium_verylow_guide:
            return "very low"
        elif X <= magnesium_low_guide:
            return "low"
        elif X <= magnesium_high_guide:
            return "optimum"
        else:
            return "high"
    elif chemical == 'calcium':
        calcium_veryhigh_guide, calcium_high_guide, calcium_sample_guide, calcium_low_guide, calcium_verylow_guide = cursor_lims.execute(
        f"""
        DECLARE @Critical_K FLOAT;
        select @Critical_K = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial K Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Ca FLOAT;
        select @Critical_Ca = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Ca Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @Critical_Mg FLOAT;
        select @Critical_Mg = isnull(target_value,0) from dbo.YieldTargets where crop_code = {crop_code} and target_desc like '%Critial Mg Level%' and Client_id={client_id} and Lab_Code= 0;
    
        DECLARE @interpretation_Code INT;
        select @interpretation_Code=interpretation_code from interpretations WHERE crop_code = {crop_code} AND Client_Id = {client_id} and Chemical_Code = {chemical_code};
    
        DECLARE @veryhigh_guide FLOAT;
        select @veryhigh_guide=dbo.Sample_Guides(@interpretation_Code, 0);
        select @veryhigh_guide = {cec} * @veryhigh_guide/100 *390;
    
        DECLARE @high_guide FLOAT;
        select @high_guide=dbo.Sample_Guides(@interpretation_Code, 1);
        select @high_guide = {cec} * @high_guide/100 *390;
        
        DECLARE @sample_guide FLOAT;
        select @sample_guide=dbo.Sample_Guides(@interpretation_Code, 2)
        select @sample_guide= {cec} *@sample_guide/100 *390
    
        DECLARE @low_guide FLOAT;
        select @low_guide=dbo.Sample_Guides(@interpretation_Code, 3)
        select @low_guide= {cec} * @low_guide/100 * 390
    
        DECLARE @verylow_guide FLOAT;
        select @verylow_guide=dbo.Sample_Guides(@interpretation_Code, 4)
        select @verylow_guide= {cec} * @verylow_guide/100 * 390
        
        select @veryhigh_guide, @high_guide,  @sample_guide, @low_guide, @verylow_guide
        """).fetchone()
        if X <= calcium_verylow_guide:
            return "very low"
        elif X <= calcium_low_guide:
            return "low"
        elif X <= calcium_high_guide:
            return "optimum"
        else:
            return "high"
    else:
        if X <= lim1:
            return "very low"
        elif X <= lim2:
            return "low"
        elif X <= lim3:
            return "optimum"
        else:
            return "high"

In [10]:
df = df.rename(columns={"pH":"ph", "Available P":"phosphorus","Exchangeable K":"potassium", 'C.E.C':'cec','Organic Matter': 'organic_matter'})

In [11]:
df['Crop']

0                     Maize
1                     Maize
2                     Maize
3                     Maize
4                     Maize
              ...          
70    Tomatoes (Open field)
71    Tomatoes (Open field)
72    Tomatoes (Open field)
73    Tomatoes (Open field)
74    Tomatoes (Open field)
Name: Crop, Length: 75, dtype: object

In [12]:
client = 'KALRO'

In [13]:
result_df = pd.DataFrame({"barcode":df['Sample Code']})
for index, row in df.iterrows():
    barcode = row['Sample Code'].strip()
    crop = row['Crop'].strip()
    client = client
    ph = str(row['ph']).strip()
    phosphorus = str(row['phosphorus']).strip()
    potassium = str(row['potassium']).strip()
    organic_matter = str(row['organic_matter']).strip()
    cec = str(row['cec']).strip()
    #calcium = str(row['calcium']).strip()
    # magnesium = str(row['magnesium']).strip()

    client_id = cursor_lims.execute(f"""
            DECLARE @client_id INT
            SELECT @client_id = isnull(client_id,0) from dbo.Clients WHERE client_name = '{client}'
            SELECT @client_id
        """).fetchone()
    crop_code = cursor_lims.execute(f"""
        DECLARE @crop_code INT
        SELECT @crop_code = isnull(crop_code,0) from dbo.Crops WHERE crop_name = '{crop}'
        SELECT @crop_code
    """).fetchone()
    ph_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'ph'
        SELECT @chemical_code
    """).fetchone()
    phosphorus_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'phosphorus'
        SELECT @chemical_code
    """).fetchone()
    potassium_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'potassium'
        SELECT @chemical_code
    """).fetchone()
    organic_matter_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'organic matter'
        SELECT @chemical_code
    """).fetchone()
    calcium_code = cursor_lims.execute(f"""
        DECLARE @chemical_code INT
        SELECT @chemical_code = isnull(chemical_code,0) from dbo.chemicals WHERE chemical_name = 'calcium'
        SELECT @chemical_code
    """).fetchone()

    vindexes = pd.read_sql(f"""
        SELECT vIndexes.guide, Clients.client_id, Clients.client_name, LOWER(vIndexes.status_name) AS status_name, Crops.crop_name, vIndexes.crop_code, LOWER(vIndexes.Chemical_Name) AS chemical_name, chemicals.chemical_code 
        FROM vIndexes
        INNER JOIN Crops
        ON Crops.crop_code = vIndexes.crop_code
        INNER JOIN Clients
        ON Clients.client_id = vIndexes.client_id
        INNER JOIN Chemicals
        ON Chemicals.Chemical_Name = vIndexes.Chemical_Name
        WHERE
        vIndexes.lab_code =7 and
        vIndexes.group_code =3 and
        vIndexes.growth_code =0 and
        vIndexes.crop_name = '{crop}'  AND
        Clients.client_name = '{client}' AND Clients.client_type=6 AND
        guide IS NOT NULL
        """,con=conn_lims)

    #calcium_guides = vindexes.loc[vindexes['chemical_name']=='calcium']
    #calcium_guides = calcium_guides.drop_duplicates(subset='status_name')
    #calcium_guides = calcium_guides[['status_name','guide']]
    #calcium_guides = calcium_guides.set_index('status_name')
    #calcium_guides = calcium_guides.to_dict()
    #print("Calcium guides:", calcium_guides)
    
    # magnesium_guides = vindexes.loc[vindexes['chemical_name']=='magnesium']
    # magnesium_guides = magnesium_guides.drop_duplicates(subset='status_name')
    # magnesium_guides = magnesium_guides[['status_name','guide']]
    # magnesium_guides = magnesium_guides.set_index('status_name')
    # magnesium_guides = magnesium_guides.to_dict()

    potassium_guides = vindexes.loc[vindexes['chemical_name']=='potassium']
    potassium_guides = potassium_guides.drop_duplicates(subset='status_name')
    potassium_guides = potassium_guides[['status_name','guide']]
    potassium_guides = potassium_guides.set_index('status_name')
    potassium_guides = potassium_guides.to_dict()

    organic_matter_guides = vindexes.loc[vindexes['chemical_name']=='organic matter']
    organic_matter_guides = organic_matter_guides.drop_duplicates(subset='status_name')
    organic_matter_guides = organic_matter_guides[['status_name','guide']]
    organic_matter_guides = organic_matter_guides.set_index('status_name')
    organic_matter_guides = organic_matter_guides.to_dict()
    
    ph_guides = vindexes.loc[vindexes['chemical_name']=='ph']
    ph_guides = ph_guides.drop_duplicates(subset='status_name')
    ph_guides = ph_guides[['status_name','guide']]
    ph_guides = ph_guides.set_index('status_name')
    ph_guides = ph_guides.to_dict()

    ph_class = classifyResults(float(ph.replace("<","").replace(">","")), float(cec), 'ph', lim1=ph_guides['guide']['very low'], lim2=ph_guides['guide']['low'], lim3=ph_guides['guide']['high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=ph_code[0])
    potassium_class = classifyResults(float(potassium.replace("<","").replace(">","")), float(cec), 'potassium', lim1=potassium_guides['guide']['very low'], lim2=potassium_guides['guide']['low'], lim3=potassium_guides['guide']['high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=potassium_code[0])
    organic_matter_class = classifyResults(float(organic_matter.replace("<","").replace(">","")), float(cec), 'organic_matter', lim1=organic_matter_guides['guide']['very low'], lim2=organic_matter_guides['guide']['low'], lim3=organic_matter_guides['guide']['high'], crop_code=crop_code[0], client_id=client_id[0], chemical_code=organic_matter_code[0])

    comparison_df_ = comparison_df.loc[comparison_df['barcode'] == barcode]
    if comparison_df_['ph'].values[0] in ['very low','low','high','optimum'] and comparison_df_['ph'].values[0] == ph_class:
        result_df.loc[result_df['barcode'] == barcode, 'ph'] = None
    elif comparison_df_['ph'].values[0] in ['very low','low','high','optimum'] and comparison_df_['ph'].values[0] != ph_class:
        result_df.loc[result_df['barcode'] == barcode, 'ph'] = str(comparison_df_['ph'].values[0]) +'/'+ str(ph_class) +":"+ ph
    if comparison_df_['potassium'].values[0] in ['very low','low','high','optimum'] and comparison_df_['potassium'].values[0] == potassium_class:
        result_df.loc[result_df['barcode'] == barcode, 'potassium'] = None
    elif comparison_df_['potassium'].values[0] in ['very low','low','high','optimum'] and comparison_df_['potassium'].values[0] != potassium_class:
        result_df.loc[result_df['barcode'] == barcode, 'potassium'] = str(comparison_df_['potassium'].values[0]) +'/'+ str(potassium_class) +":"+ potassium
    if comparison_df_['Organic Matter'].values[0] in ['very low','low','high','optimum'] and comparison_df_['Organic Matter'].values[0] == organic_matter_class:
        result_df.loc[result_df['barcode'] == barcode, 'organic_matter'] = None
    elif comparison_df_['Organic Matter'].values[0] in ['very low','low','high','optimum'] and comparison_df_['Organic Matter'].values[0] != organic_matter_class:
        result_df.loc[result_df['barcode'] == barcode, 'organic_matter'] = str(comparison_df_['Organic Matter'].values[0]) +'/'+ str(organic_matter_class) +":"+ organic_matter

C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_34216\3576808930.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vindexes = pd.read_sql(f"""
C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_34216\3576808930.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vindexes = pd.read_sql(f"""
C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_34216\3576808930.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vindexes = pd.read_sql(f"""
C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_34216\3576808930.py:50: UserWar

In [14]:
result_df

,barcode,ph,potassium,organic_matter
0,TEST-DS1-1660,None,None,None
1,TEST-DS1-1661,None,None,None
2,TEST-DS1-1663,None,None,None
3,TEST-DS1-1664,None,None,None
4,TEST-DS1-1665,None,None,None
...,...,...,...,...
70,TEST-DS1-1767,None,None,None
71,TEST-DS1-1768,None,None,None
72,TEST-DS1-1769,None,None,None
73,TEST-DS1-1770,None,low/optimum:140.0,None


In [15]:
result_df.ph.value_counts()

Series([], Name: count, dtype: int64)

In [16]:
result_df.potassium.value_counts()

potassium
optimum/high:740.0    1
low/optimum:90.0      1
low/optimum:140.0     1
Name: count, dtype: int64

In [17]:
result_df.organic_matter.value_counts()

Series([], Name: count, dtype: int64)

In [18]:
result_df.to_csv("outputFiles/verification_compared_guides.csv")